In [ ]:
import numpy as np
import pandas as pd
import torch
from folktables import ACSDataSource, ACSIncome
import deepchem as dc
from sklearn.linear_model import LogisticRegression
import os

# 1. ACS (California Data for Classifier-Training, Evaluate on Puerto Rico)

In [ ]:
'''
Reference: https://github.com/socialfoundations/folktables
*Directly used their "Distribution shift across states" vignette. Just swapping Michigan for Puerto Rico
'''
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
ca_data = data_source.get_data(states=["CA"], download=True)
pr_data = data_source.get_data(states=["PR"], download=True)
ca_features, ca_labels, _ = ACSIncome.df_to_numpy(ca_data)
pr_features, pr_labels, _ = ACSIncome.df_to_numpy(pr_data)

# standardize our features
ca_mean, ca_std = ca_features.mean(axis=0), ca_features.std(axis=0)
ca_features = (ca_features - ca_mean) / ca_std
pr_features = (pr_features - ca_mean) / ca_std

# Plug-in LogisticRegression method for tabular datasets
model = LogisticRegression(fit_intercept=True)

# Train on CA data
model.fit(ca_features, ca_labels)

# save testing data, and initial coefficients (incl. intercept) for the logistic regression
pr_features = np.hstack([np.ones(shape=(pr_features.shape[0], 1)), pr_features])
torch.save(torch.tensor(pr_features).to(torch.float32), "datasets/acs/X_test.pt")
torch.save(torch.tensor(pr_labels).to(torch.float32), "datasets/acs/y_test.pt")
beta_init = np.hstack([model.intercept_[None,:], model.coef_])[0]
torch.save(torch.tensor(beta_init).to(torch.float32), "datasets/acs/beta_init.pt")

# 2. AlphaFold (from PPI/Zrnic and Candès)

In [ ]:
'''
Reference: Prediction-Powered Inference (https://zenodo.org/records/8397451)
- Y: indicator that a residue is disordered (i.e., in an intrinsically disordered region).
- Yhat: AlphaFold-derived predicted probability that the residue is disordered.
'''
# load in our data
data = np.load("datasets/alphafold/alphafold.npz")

# save in the right formats
torch.save(torch.tensor(data["Y"]).to(torch.float32), "datasets/alphafold/y_test.pt")
torch.save(torch.tensor(data["Yhat"]).to(torch.float32), "datasets/alphafold/fhat_test.pt")

# 3. BODMAS

In [ ]:
# source for BODMAS dataset: https://whyisyoung.github.io/BODMAS/, sorted in chronological order.
data = np.load("datasets/bodmas/bodmas.npz")
X = data['X']  # all the feature vectors
y = data['y']  # labels, 0 as benign, 1 as malicious

# split chronologically: 50/50 train/test split
X_train, y_train = X[:X.shape[0] // 2], y[:X.shape[0] // 2]
X_test, y_test = X[X.shape[0] // 2:], y[X.shape[0] // 2:]

# standardize our data
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)

# only keep features that don't have STD 0
X_train, X_test = X_train[:,X_std != 0], X_test[:,X_std != 0]
X_mean, X_std = X_mean[X_std != 0], X_std[X_std != 0]

# standardize both train and test using the same training mean & SD
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

# Plug-in LogisticRegression method for tabular datasets
model = LogisticRegression(fit_intercept=True, max_iter=1000)

# Train on the training set
model.fit(X_train, y_train)

# save testing data, and initial coefficients (incl. intercept) for the logistic regression
X_test = np.hstack([np.ones(shape=(X_test.shape[0], 1)), X_test])
torch.save(torch.tensor(X_test).to(torch.float32), "datasets/bodmas/X_test.pt")
torch.save(torch.tensor(y_test).to(torch.float32), "datasets/bodmas/y_test.pt")
beta_init = np.hstack([model.intercept_[None,:], model.coef_])[0]
torch.save(torch.tensor(beta_init).to(torch.float32), "datasets/bodmas/beta_init.pt")

# 4. HIV

In [ ]:
# see https://deepchem.readthedocs.io/en/latest/api_reference/moleculenet.html#hiv-datasets
tasks, datasets, _ = dc.molnet.load_hiv(
    featurizer="ECFP",
    splitter=None,
    transformers=[],
)
dataset = datasets[0]
X = np.asarray(dataset.X, dtype=np.float32)
y = np.asarray(dataset.y[:, 0], dtype=np.int32)

# split chronologically: 50/50 train/test split
X_train, y_train = X[:X.shape[0] // 2], y[:X.shape[0] // 2]
X_test, y_test = X[X.shape[0] // 2:], y[X.shape[0] // 2:]

# standardize our data
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)

# only keep features that don't have STD 0
X_train, X_test = X_train[:,X_std != 0], X_test[:,X_std != 0]
X_mean, X_std = X_mean[X_std != 0], X_std[X_std != 0]

# standardize both train and test using the same training mean & SD
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

# Plug-in LogisticRegression method for tabular datasets
model = LogisticRegression(fit_intercept=True, max_iter=1000)

# Train on the training set
model.fit(X_train, y_train)

# save testing data, and initial coefficients (incl. intercept) for the logistic regression
X_test = np.hstack([np.ones(shape=(X_test.shape[0], 1)), X_test])
torch.save(torch.tensor(X_test).to(torch.float32), "datasets/hiv/X_test.pt")
torch.save(torch.tensor(y_test).to(torch.float32), "datasets/hiv/y_test.pt")
beta_init = np.hstack([model.intercept_[None,:], model.coef_])[0]
torch.save(torch.tensor(beta_init).to(torch.float32), "datasets/hiv/beta_init.pt")

# 5. LLM Evaluations

In [ ]:
'''
1. Only need to do fully-observed historical data -- convert everything into pytorch:
    - URL: https://github.com/skbwu/efficiently-evaluating-llms/tree/main/data

2. Download history-fitted model and question factors direct from:
    - URL: https://github.com/skbwu/efficiently-evaluating-llms/tree/main/factor_models/final
'''
# start with mmlu-pro
M2 = pd.read_csv("datasets/llm_evals/mmlu-pro/M2.csv")
torch.save(
    torch.tensor(M2.iloc[:,3:].to_numpy()).to(torch.float32), 
    "datasets/llm_evals/mmlu-pro/Y_test.pt")

# also do bbh+gpqa+ifeval+math+musr
M2 = pd.read_csv("datasets/llm_evals/bbh+gpqa+ifeval+math+musr/M2.csv")
torch.save(
    torch.tensor(M2.iloc[:,3:].to_numpy()).to(torch.float32), 
    "datasets/llm_evals/bbh+gpqa+ifeval+math+musr/Y_test.pt")

# 6. Pew

In [ ]:
# y_i: did participant lose money to a scam?
LABEL = "OSCAM_MONEY1_W168" # "Have you ever lost money because of an online scam or attack?"

# various relevant covariates from the polling to extract from the raw data file
FEATURES = [
    # ATP profile/demographic info
    "F_AGECAT",
    "F_GENDER",
    "F_EDUCCAT",
    "F_RACETHNMOD",
    "F_INC_TIER2",
    "F_MARITAL",
    "F_CREGION",
    "F_METRO",
    "F_INTFREQ",

    # Online behavior
    "SNSUSE_W168",
    "ONLTSK_a_W168",
    "ONLTSK_b_W168",

    # Scam awareness
    "OSCAM_AWARE_W168",
    "OSCAM_PROB_W168",
    "OSCAM_KNOW_W168",

    # Friends/family scam experience
    "OSCAM_TYPEOTHER1_a_W168",
    "OSCAM_TYPEOTHER1_b_W168",
    "OSCAM_TYPEOTHER2_a_W168",
    "OSCAM_TYPEOTHER2_b_W168",
    "OSCAM_TYPEOTHER2_c_W168",
    "OSCAM_TYPEOTHER3_a_W168",
    "OSCAM_TYPEOTHER3_b_W168",
]

# see https://www.pewresearch.org/dataset/american-trends-panel-wave-168/
df = pd.read_csv("datasets/pew/W168_Apr25/ATP W168.csv")[[LABEL] + FEATURES].dropna()

# 99 refers to "refuse to answer" -- let's drop those
df = df[~(df == 99).any(axis=1)].astype(int)

# y_i : OSCAM_MONEY1_W168. 1=yes, 2=no -> recode as 0
y = df[LABEL].replace(2, 0).values

# one-hot encode all of our features, which are categorical
X = pd.get_dummies(
    df[FEATURES], columns=FEATURES, dtype=float, drop_first=True).to_numpy()

# split chronologically: 50/50 train/test split
X_train, y_train = X[:X.shape[0] // 2], y[:X.shape[0] // 2]
X_test, y_test = X[X.shape[0] // 2:], y[X.shape[0] // 2:]

# standardize our data
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)

# only keep features that don't have STD 0
X_train, X_test = X_train[:,X_std != 0], X_test[:,X_std != 0]
X_mean, X_std = X_mean[X_std != 0], X_std[X_std != 0]

# standardize both train and test using the same training mean & SD
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

# Plug-in LogisticRegression method for tabular datasets
model = LogisticRegression(fit_intercept=True, max_iter=1000)

# Train on the training set
model.fit(X_train, y_train)

# save testing data, and initial coefficients (incl. intercept) for the logistic regression
X_test = np.hstack([np.ones(shape=(X_test.shape[0], 1)), X_test])
torch.save(torch.tensor(X_test).to(torch.float32), "datasets/pew/X_test.pt")
torch.save(torch.tensor(y_test).to(torch.float32), "datasets/pew/y_test.pt")
beta_init = np.hstack([model.intercept_[None,:], model.coef_])[0]
torch.save(torch.tensor(beta_init).to(torch.float32), "datasets/pew/beta_init.pt")

# 7. ULB

In [ ]:
# source for ULB Credit Card Fraud: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download
df = pd.read_csv("datasets/ulb/creditcard.csv")

# extract the X's and y's
X, y = df.to_numpy()[:,1:-1], df.Class.values

# split chronologically: 50/50 train/test split
X_train, y_train = X[:X.shape[0] // 2], y[:X.shape[0] // 2]
X_test, y_test = X[X.shape[0] // 2:], y[X.shape[0] // 2:]

# standardize our data
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)

# only keep features that don't have STD 0
X_train, X_test = X_train[:,X_std != 0], X_test[:,X_std != 0]
X_mean, X_std = X_mean[X_std != 0], X_std[X_std != 0]

# standardize both train and test using the same training mean & SD
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

# Plug-in LogisticRegression method for tabular datasets
model = LogisticRegression(fit_intercept=True, max_iter=1000)

# Train on the training set
model.fit(X_train, y_train)

# save testing data, and initial coefficients (incl. intercept) for the logistic regression
X_test = np.hstack([np.ones(shape=(X_test.shape[0], 1)), X_test])
torch.save(torch.tensor(X_test).to(torch.float32), "datasets/ulb/X_test.pt")
torch.save(torch.tensor(y_test).to(torch.float32), "datasets/ulb/y_test.pt")
beta_init = np.hstack([model.intercept_[None,:], model.coef_])[0]
torch.save(torch.tensor(beta_init).to(torch.float32), "datasets/ulb/beta_init.pt")